## bm-f

In [2]:
import torch
from compressai.models import DCAE, TCM
import compressai.runtime.utils.metrics as metrics
import numpy as np
import torch.nn.functional as F
from compressai.utils import dataloader_science
from compressai.zoo import bmshj2018_factorized

from compressai.runtime import build_runtime
from compressai.runtime.config import RuntimeConfig
from compressai.runtime.codecs import GpuPackedEntropyCodec
from compressai.runtime.utils.benchmark import run_e2e

# config
quality = 8
dataset = "hurricane"
bin_path = "/hwj/data/caiec_test_data/hurricane_100x1x500x500.f32"
input_shape = (100, 500, 500)            # (N,H,W)
block_size = (3, 512, 512)            # (bn,3,bh,bw)
p=0.95
stride = (int(block_size[1] * p), int(block_size[2] * p))
device = "cuda:0"
batchsize = 34
norm_type = "minmax"
pad_value = 0.5

blocks, x_nhw_ori, status, meta = dataloader_science.blockify_bin_overlap(
    bin_path, input_shape, block_size, p, stride_hw=stride,
    dtype=np.float32, pad_value=pad_value, norm_type=norm_type, device=device
)
bn = blocks.shape[0]

net = bmshj2018_factorized(quality=quality)
net = net.to(device)
net.eval()

checkpoint = torch.load(f"/hwj/data/bmshj2018-factorized/bmshj2018-factorized-{quality}.pth", map_location=device)
# for k, v in checkpoint["state_dict"].items():
#     dictory[k.replace("module.", "")] = v
net.load_state_dict(checkpoint)
net.update()   

codec = GpuPackedEntropyCodec(
    net.entropy_bottleneck,
    P=1
)

cfg = RuntimeConfig(
    model_name="bmshj2018_factorized",
    ga_input_dtype=torch.float32,
    gs_input_dtype=torch.float16,
    codec_input_dtype=torch.float32,
    trt_engines={
        "ga":f"/hwj/data/engines/bmshj2018-factorized/q{quality}/engines/{dataset}/ga/fp8.engine",
        "gs":f"/hwj/data/engines/bmshj2018-factorized/q{quality}/engines/{dataset}/gs/fp16.engine",
    },
)

engine = build_runtime(net, codec, cfg)

xhat_blocks = torch.empty_like(blocks)
bit_len = 0
strings_bytes_list = []
state_bytes_list = []

cmp_time_lists = []
decmp_time_lists = []

enc_start = torch.cuda.Event(enable_timing=True)
enc_end   = torch.cuda.Event(enable_timing=True)
dec_start = torch.cuda.Event(enable_timing=True)
dec_end   = torch.cuda.Event(enable_timing=True)

input = blocks[:batchsize]  
# warmup
for i in range(5):
    pack = engine.compress(input)
torch.cuda.synchronize()

# inference
with torch.no_grad():
    for start in range(0, bn, batchsize):
        end = min(start + batchsize, bn)
        print(f"Processing blocks {start} to {end} / {bn}")
        input = blocks[start:end]   
        cur_bs = input.shape[0]
        if cur_bs < batchsize:
            pad_bs = batchsize - cur_bs
            pad_shape = (pad_bs,) + input.shape[1:]
            pad = torch.full(
                pad_shape,
                0.5,
                dtype=input.dtype,
                device=input.device,
            )
            input = torch.cat([input, pad], dim=0)
            print("input pad to:", input.shape)   

        enc_start.record()
        pack,_,_ = engine.compress_time(input)
        enc_end.record()
        torch.cuda.synchronize()
        cmp_time_lists.append(enc_start.elapsed_time(enc_end))
        
        dec_start.record()
        x_hat,_,_ = engine.decompress_time(pack)
        dec_end.record()
        torch.cuda.synchronize()
        decmp_time_lists.append(dec_start.elapsed_time(dec_end))
        
        if hasattr(codec, "pack_bytes"):
            b = codec.pack_bytes(pack)
            strings_bytes_list.append(float(b.get("strings_bytes", 0.0)))
            state_bytes_list.append(float(b.get("state_bytes", 0.0)))
        xhat_blocks[start:end] = x_hat[:cur_bs]
        del input, pack, x_hat
        torch.cuda.empty_cache()
        
assert xhat_blocks.shape[0] == bn
strings_bytes = sum(strings_bytes_list)
state_bytes = sum(state_bytes_list)
bit_len = strings_bytes + state_bytes
x_hat_nhw = dataloader_science.blocks_to_nhw_overlap_weighted(
    xhat_blocks,
    input_shape,
    block_size,
    stride_hw=stride,
    meta_hw=meta,
    pad_value=pad_value,
    norm_type="minmax",
    vmin=status["vmin"],
    vmax=status["vmax"],
    vmean=status["vmean"],
    vstd=status["vstd"],
)

# compute metrics
metr = metrics.basic_metrics(x_hat_nhw, x_nhw_ori)
num_pixels = x_nhw_ori.size(0) * x_nhw_ori.size(1) * x_nhw_ori.size(2)
bit_rate = bit_len * 8.0 / num_pixels

# thr
ori_size = x_nhw_ori.numel() * x_nhw_ori.element_size()
cmpthr = ori_size / 1024 / 1024 / 1024 / (sum(cmp_time_lists) / 1000)
decmpthr = ori_size / 1024 / 1024 / 1024 / (sum(decmp_time_lists) / 1000)

print("bpp:", bit_rate)
print("rmse:", metr["rmse"])
print("nrmse:", metr["nrmse"])
print("maxe:", metr["maxe"])
print("psnr:", metr["psnr"])
print("Compress Throughput:", cmpthr, "GB/s")
print("Decompress Throughput:", decmpthr, "GB/s")

[01/27/2026-12:57:58] [TRT] [W] Using default stream in enqueueV3() may lead to performance issues due to additional calls to cudaStreamSynchronize() by TensorRT to ensure correct synchronization. Please use non-default stream instead.
Processing blocks 0 to 34 / 34
[01/27/2026-12:57:58] [TRT] [W] Using default stream in enqueueV3() may lead to performance issues due to additional calls to cudaStreamSynchronize() by TensorRT to ensure correct synchronization. Please use non-default stream instead.
bpp: 0.24765696
rmse: 22.46448516845703
nrmse: 0.00973963551223278
maxe: 1573.939453125
psnr: 40.19438171386719
Compress Throughput: 8.8627162809068 GB/s
Decompress Throughput: 4.549094138374727 GB/s


## bm-h

In [4]:
import torch
from compressai.models import DCAE, TCM
import compressai.runtime.utils.metrics as metrics
import numpy as np
import torch.nn.functional as F
from compressai.utils import dataloader_science
from compressai.zoo import bmshj2018_factorized, bmshj2018_hyperprior

from compressai.runtime import build_runtime
from compressai.runtime.config import RuntimeConfig
from compressai.runtime.codecs import GpuPackedEntropyCodec
from compressai.runtime.utils.benchmark import run_e2e

# config
quality = 8
dataset = "hurricane"
bin_path = "/hwj/data/caiec_test_data/hurricane_100x1x500x500.f32"
input_shape = (100, 500, 500)            # (N,H,W)
block_size = (3, 512, 512)            # (bn,3,bh,bw)
p=0.95
stride = (int(block_size[1] * p), int(block_size[2] * p))
device = "cuda:0"
batchsize = 34
norm_type = "minmax"
pad_value = 0.5

blocks, x_nhw_ori, status, meta = dataloader_science.blockify_bin_overlap(
    bin_path, input_shape, block_size, p, stride_hw=stride,
    dtype=np.float32, pad_value=pad_value, norm_type=norm_type, device=device
)
bn = blocks.shape[0]

net = bmshj2018_hyperprior(quality=quality)
net = net.to(device)
net.eval()

checkpoint = torch.load(f"/hwj/data/bmshj2018-hyperprior/bmshj2018-hyperprior-{quality}.pth", map_location=device)
# for k, v in checkpoint["state_dict"].items():
#     dictory[k.replace("module.", "")] = v
net.load_state_dict(checkpoint)
net.update()   

codec = GpuPackedEntropyCodec(
    net.entropy_bottleneck,
    net.gaussian_conditional,
    P=1
)

cfg = RuntimeConfig(
    model_name="bmshj2018_hyperprior",
    ga_input_dtype=torch.float32,
    gs_input_dtype=torch.float16,
    ha_input_dtype=torch.float32,
    hs_input_dtype=torch.float32,
    codec_input_dtype=torch.float32,
    trt_engines={
        "ga":f"/hwj/data/engines/bmshj2018-hyperprior/q{quality}/engines/{dataset}/ga/fp8.engine",
        "gs":f"/hwj/data/engines/bmshj2018-hyperprior/q{quality}/engines/{dataset}/gs/fp16.engine",
        "ha":f"/hwj/data/engines/bmshj2018-hyperprior/q{quality}/engines/{dataset}/ha/fp8.engine",
        "hs":f"/hwj/data/engines/bmshj2018-hyperprior/q{quality}/engines/{dataset}/hs/fp8.engine",
        
    },
)

engine = build_runtime(net, codec, cfg)

xhat_blocks = torch.empty_like(blocks)
bit_len = 0
strings_bytes_list = []
state_bytes_list = []

cmp_time_lists = []
decmp_time_lists = []

enc_start = torch.cuda.Event(enable_timing=True)
enc_end   = torch.cuda.Event(enable_timing=True)
dec_start = torch.cuda.Event(enable_timing=True)
dec_end   = torch.cuda.Event(enable_timing=True)

input = blocks[:batchsize]  
# warmup
for i in range(5):
    pack = engine.compress(input)
torch.cuda.synchronize()

# inference
with torch.no_grad():
    for start in range(0, bn, batchsize):
        end = min(start + batchsize, bn)
        print(f"Processing blocks {start} to {end} / {bn}")
        input = blocks[start:end]   
        cur_bs = input.shape[0]
        if cur_bs < batchsize:
            pad_bs = batchsize - cur_bs
            pad_shape = (pad_bs,) + input.shape[1:]
            pad = torch.full(
                pad_shape,
                0.5,
                dtype=input.dtype,
                device=input.device,
            )
            input = torch.cat([input, pad], dim=0)
            print("input pad to:", input.shape)   

        enc_start.record()
        pack,_,_ = engine.compress_time(input)
        enc_end.record()
        torch.cuda.synchronize()
        cmp_time_lists.append(enc_start.elapsed_time(enc_end))
        
        dec_start.record()
        x_hat,_,_ = engine.decompress_time(pack)
        dec_end.record()
        torch.cuda.synchronize()
        decmp_time_lists.append(dec_start.elapsed_time(dec_end))
        
        if hasattr(codec, "pack_bytes"):
            b = codec.pack_bytes(pack)
            strings_bytes_list.append(float(b.get("strings_bytes", 0.0)))
            state_bytes_list.append(float(b.get("state_bytes", 0.0)))
        xhat_blocks[start:end] = x_hat[:cur_bs]
        del input, pack, x_hat
        torch.cuda.empty_cache()
        
assert xhat_blocks.shape[0] == bn
strings_bytes = sum(strings_bytes_list)
state_bytes = sum(state_bytes_list)
bit_len = strings_bytes + state_bytes
x_hat_nhw = dataloader_science.blocks_to_nhw_overlap_weighted(
    xhat_blocks,
    input_shape,
    block_size,
    stride_hw=stride,
    meta_hw=meta,
    pad_value=pad_value,
    norm_type="minmax",
    vmin=status["vmin"],
    vmax=status["vmax"],
    vmean=status["vmean"],
    vstd=status["vstd"],
)

# compute metrics
metr = metrics.basic_metrics(x_hat_nhw, x_nhw_ori)
num_pixels = x_nhw_ori.size(0) * x_nhw_ori.size(1) * x_nhw_ori.size(2)
bit_rate = bit_len * 8.0 / num_pixels

# thr
ori_size = x_nhw_ori.numel() * x_nhw_ori.element_size()
cmpthr = ori_size / 1024 / 1024 / 1024 / (sum(cmp_time_lists) / 1000)
decmpthr = ori_size / 1024 / 1024 / 1024 / (sum(decmp_time_lists) / 1000)

print("bpp:", bit_rate)
print("rmse:", metr["rmse"])
print("nrmse:", metr["nrmse"])
print("maxe:", metr["maxe"])
print("psnr:", metr["psnr"])
print("Compress Throughput:", cmpthr, "GB/s")
print("Decompress Throughput:", decmpthr, "GB/s")

[01/27/2026-12:58:35] [TRT] [W] Using default stream in enqueueV3() may lead to performance issues due to additional calls to cudaStreamSynchronize() by TensorRT to ensure correct synchronization. Please use non-default stream instead.
[01/27/2026-12:58:35] [TRT] [W] Using default stream in enqueueV3() may lead to performance issues due to additional calls to cudaStreamSynchronize() by TensorRT to ensure correct synchronization. Please use non-default stream instead.
[01/27/2026-12:58:35] [TRT] [W] Using default stream in enqueueV3() may lead to performance issues due to additional calls to cudaStreamSynchronize() by TensorRT to ensure correct synchronization. Please use non-default stream instead.
Processing blocks 0 to 34 / 34
[01/27/2026-12:58:35] [TRT] [W] Using default stream in enqueueV3() may lead to performance issues due to additional calls to cudaStreamSynchronize() by TensorRT to ensure correct synchronization. Please use non-default stream instead.
bpp: 0.08996992
rmse: 19.

## mbt2018-mean

In [6]:
import torch
from compressai.models import DCAE, TCM
import compressai.runtime.utils.metrics as metrics
import numpy as np
import torch.nn.functional as F
from compressai.utils import dataloader_science
from compressai.zoo import mbt2018_mean

from compressai.runtime import build_runtime
from compressai.runtime.config import RuntimeConfig
from compressai.runtime.codecs import GpuPackedEntropyCodec
from compressai.runtime.utils.benchmark import run_e2e

# config
quality = 8
dataset = "hurricane"
bin_path = "/hwj/data/caiec_test_data/hurricane_100x1x500x500.f32"
input_shape = (100, 500, 500)            # (N,H,W)
block_size = (3, 512, 512)            # (bn,3,bh,bw)
p=0.95
stride = (int(block_size[1] * p), int(block_size[2] * p))
device = "cuda:0"
batchsize = 34
norm_type = "minmax"
pad_value = 0.5

blocks, x_nhw_ori, status, meta = dataloader_science.blockify_bin_overlap(
    bin_path, input_shape, block_size, p, stride_hw=stride,
    dtype=np.float32, pad_value=pad_value, norm_type=norm_type, device=device
)
bn = blocks.shape[0]

net = mbt2018_mean(quality=quality)
net = net.to(device)
net.eval()

checkpoint = torch.load(f"/hwj/data/mbt2018-mean/mbt2018-mean-{quality}.pth", map_location=device)
# for k, v in checkpoint["state_dict"].items():
#     dictory[k.replace("module.", "")] = v
net.load_state_dict(checkpoint)
net.update()   

codec = GpuPackedEntropyCodec(
    net.entropy_bottleneck,
    net.gaussian_conditional,
    P=12
)

cfg = RuntimeConfig(
    model_name="mbt2018_mean",
    ga_input_dtype=torch.float32,
    gs_input_dtype=torch.float16,
    ha_input_dtype=torch.float32,
    hs_input_dtype=torch.float32,
    codec_input_dtype=torch.float32,
    trt_engines={
        "ga":f"/hwj/data/engines/mbt2018-mean/q{quality}/engines/{dataset}/ga/fp8.engine",
        "gs":f"/hwj/data/engines/mbt2018-mean/q{quality}/engines/{dataset}/gs/fp16.engine",
        "ha":f"/hwj/data/engines/mbt2018-mean/q{quality}/engines/{dataset}/ha/fp8.engine",
        "hs":f"/hwj/data/engines/mbt2018-mean/q{quality}/engines/{dataset}/hs/fp8.engine",
    }
)

engine = build_runtime(net, codec, cfg)

xhat_blocks = torch.empty_like(blocks)
bit_len = 0
strings_bytes_list = []
state_bytes_list = []

# warmup

with torch.no_grad():
    for start in range(0, bn, batchsize):
        end = min(start + batchsize, bn)
        
        print(f"Processing blocks {start} to {end} / {bn}")
        input = blocks[start:end]   
        cur_bs = input.shape[0]

        if cur_bs < batchsize:
            pad_bs = batchsize - cur_bs
            pad_shape = (pad_bs,) + input.shape[1:]
            pad = torch.full(
                pad_shape,
                0.5,
                dtype=input.dtype,
                device=input.device,
            )
            input = torch.cat([input, pad], dim=0)
            print("input pad to:", input.shape)   
              
        # print("g_a[0, 0, 0, :10]", y[0, 0, 0, :10])
        pack,_,_ = engine.compress_time(input)
        x_hat,_,_ = engine.decompress_time(pack)
        if hasattr(codec, "pack_bytes"):
            b = codec.pack_bytes(pack)
            strings_bytes_list.append(float(b.get("strings_bytes", 0.0)))
            state_bytes_list.append(float(b.get("state_bytes", 0.0)))


        xhat_blocks[start:end] = x_hat[:cur_bs]
        del input, pack, x_hat
        torch.cuda.empty_cache()
        
assert xhat_blocks.shape[0] == bn
strings_bytes = sum(strings_bytes_list)
state_bytes = sum(state_bytes_list)
bit_len = strings_bytes + state_bytes
x_hat_nhw = dataloader_science.blocks_to_nhw_overlap_weighted(
    xhat_blocks,
    input_shape,
    block_size,
    stride_hw=stride,
    meta_hw=meta,
    pad_value=pad_value,
    norm_type="minmax",
    vmin=status["vmin"],
    vmax=status["vmax"],
    vmean=status["vmean"],
    vstd=status["vstd"],
)


# compute metrics
metr = metrics.basic_metrics(x_hat_nhw, x_nhw_ori)
num_pixels = x_nhw_ori.size(0) * x_nhw_ori.size(1) * x_nhw_ori.size(2)
bit_rate = bit_len * 8.0 / num_pixels

print("bpp:", bit_rate)
print("rmse:", metr["rmse"])
print("nrmse:", metr["nrmse"])
print("maxe:", metr["maxe"])
print("psnr:", metr["psnr"])

Processing blocks 0 to 34 / 34
[01/27/2026-12:59:07] [TRT] [W] Using default stream in enqueueV3() may lead to performance issues due to additional calls to cudaStreamSynchronize() by TensorRT to ensure correct synchronization. Please use non-default stream instead.
[01/27/2026-12:59:07] [TRT] [W] Using default stream in enqueueV3() may lead to performance issues due to additional calls to cudaStreamSynchronize() by TensorRT to ensure correct synchronization. Please use non-default stream instead.
[01/27/2026-12:59:07] [TRT] [W] Using default stream in enqueueV3() may lead to performance issues due to additional calls to cudaStreamSynchronize() by TensorRT to ensure correct synchronization. Please use non-default stream instead.
[01/27/2026-12:59:07] [TRT] [W] Using default stream in enqueueV3() may lead to performance issues due to additional calls to cudaStreamSynchronize() by TensorRT to ensure correct synchronization. Please use non-default stream instead.
bpp: 0.06295296
rmse: 19.

## TCM

In [ ]:
import torch
import compressai.runtime.utils.metrics as metrics
import numpy as np
import torch.nn.functional as F
from compressai.utils import dataloader_science
from compressai.models import TCM

from compressai.runtime import build_runtime
from compressai.runtime.config import RuntimeConfig
from compressai.runtime.codecs import GpuPackedEntropyCodec
from compressai.runtime.utils.benchmark import run_e2e

# config
quality = 4
dataset = "tomobank"
bin_path = "/hwj/data/caiec_test_data/tomobank_40x1x2048x2048.f32"
input_shape = (40, 2048, 2048)            # (N,H,W)
block_size = (3, 2048, 2048)            # (bn,3,bh,bw)
p=0.95
stride = (int(block_size[1] * p), int(block_size[2] * p))
device = "cuda:0"
batchsize = 3
norm_type = "minmax"
pad_value = 0.5

blocks, x_nhw_ori, status, meta = dataloader_science.blockify_bin_overlap(
    bin_path, input_shape, block_size, p, stride_hw=stride,
    dtype=np.float32, pad_value=pad_value, norm_type=norm_type, device=device
)
bn = blocks.shape[0]

net = TCM(config=[2,2,2,2,2,2], head_dim=[8, 16, 32, 32, 16, 8], drop_path_rate=0.0, N=64, M=320)
net = net.to(device)
net.eval()
dictory = {}

checkpoint = torch.load(f"/hwj/data/tcm/tcm-{quality}.pth", map_location=device)
for k, v in checkpoint["state_dict"].items():
    dictory[k.replace("module.", "")] = v
net.load_state_dict(dictory)
net.update() 

codec = GpuPackedEntropyCodec(
    net.entropy_bottleneck,
    net.gaussian_conditional,
    P=16
)

cfg = RuntimeConfig(
    model_name="tcm",
    ga_input_dtype=torch.float16,
    gs_input_dtype=torch.float16,
    ha_input_dtype=torch.float32,
    h_mean_s_input_dtype=torch.float32,
    h_scale_s_input_dtype=torch.float16,

    atten_mean_input_dtypes = [torch.float32, torch.float32, torch.float32, torch.float32, torch.float32] ,
    atten_scale_input_dtypes = [torch.float32, torch.float32, torch.float32, torch.float32, torch.float32],
    cc_mean_input_dtypes = [torch.float32, torch.float32, torch.float32, torch.float32, torch.float32],
    cc_scale_input_dtypes = [torch.float32, torch.float32, torch.float32, torch.float32, torch.float32],
    lrp_input_dtypes = [torch.float32, torch.float32, torch.float32, torch.float32, torch.float32],
    
    codec_input_dtype=torch.float32,
    trt_engines={
        "ga": f"/hwj/data/engines/tcm/q{quality}/engines/{dataset}/ga/fp16.engine",
        "gs": f"/hwj/data/engines/tcm/q{quality}/engines/{dataset}/gs/fp16.engine",
        "ha": f"/hwj/data/engines/tcm/q{quality}/engines/{dataset}/ha/fp8.engine",
        "h_mean_s": f"/hwj/data/engines/tcm/q{quality}/engines/{dataset}/h_mean_s/fp8.engine",
        "h_scale_s": f"/hwj/data/engines/tcm/q{quality}/engines/{dataset}/h_scale_s/fp16.engine",

        "atten_mean_0": f"/hwj/data/engines/tcm/q{quality}/engines/{dataset}/atten_mean_0/fp8.engine",
        "atten_scale_0": f"/hwj/data/engines/tcm/q{quality}/engines/{dataset}/atten_scale_0/fp8.engine",
        "cc_mean_0": f"/hwj/data/engines/tcm/q{quality}/engines/{dataset}/cc_mean_0/fp8.engine",
        "cc_scale_0": f"/hwj/data/engines/tcm/q{quality}/engines/{dataset}/cc_scale_0/fp8.engine",
        "lrp_transforms_0": f"/hwj/data/engines/tcm/q{quality}/engines/{dataset}/lrp_transforms_0/fp8.engine",

        "atten_mean_1": f"/hwj/data/engines/tcm/q{quality}/engines/{dataset}/atten_mean_1/fp8.engine",
        "atten_scale_1": f"/hwj/data/engines/tcm/q{quality}/engines/{dataset}/atten_scale_1/fp8.engine",
        "cc_mean_1": f"/hwj/data/engines/tcm/q{quality}/engines/{dataset}/cc_mean_1/fp8.engine",
        "cc_scale_1": f"/hwj/data/engines/tcm/q{quality}/engines/{dataset}/cc_scale_1/fp8.engine",
        "lrp_transforms_1": f"/hwj/data/engines/tcm/q{quality}/engines/{dataset}/lrp_transforms_1/fp8.engine",

        "atten_mean_2": f"/hwj/data/engines/tcm/q{quality}/engines/{dataset}/atten_mean_2/fp8.engine",
        "atten_scale_2": f"/hwj/data/engines/tcm/q{quality}/engines/{dataset}/atten_scale_2/fp8.engine",
        "cc_mean_2": f"/hwj/data/engines/tcm/q{quality}/engines/{dataset}/cc_mean_2/fp8.engine",
        "cc_scale_2": f"/hwj/data/engines/tcm/q{quality}/engines/{dataset}/cc_scale_2/fp8.engine",
        "lrp_transforms_2": f"/hwj/data/engines/tcm/q{quality}/engines/{dataset}/lrp_transforms_2/fp8.engine",

        "atten_mean_3": f"/hwj/data/engines/tcm/q{quality}/engines/{dataset}/atten_mean_3/fp8.engine",
        "atten_scale_3": f"/hwj/data/engines/tcm/q{quality}/engines/{dataset}/atten_scale_3/fp8.engine",
        "cc_mean_3": f"/hwj/data/engines/tcm/q{quality}/engines/{dataset}/cc_mean_3/fp8.engine",
        "cc_scale_3": f"/hwj/data/engines/tcm/q{quality}/engines/{dataset}/cc_scale_3/fp8.engine",
        "lrp_transforms_3": f"/hwj/data/engines/tcm/q{quality}/engines/{dataset}/lrp_transforms_3/fp8.engine",

        "atten_mean_4": f"/hwj/data/engines/tcm/q{quality}/engines/{dataset}/atten_mean_4/fp8.engine",
        "atten_scale_4": f"/hwj/data/engines/tcm/q{quality}/engines/{dataset}/atten_scale_4/fp8.engine",
        "cc_mean_4": f"/hwj/data/engines/tcm/q{quality}/engines/{dataset}/cc_mean_4/fp8.engine",
        "cc_scale_4": f"/hwj/data/engines/tcm/q{quality}/engines/{dataset}/cc_scale_4/fp8.engine",
        "lrp_transforms_4": f"/hwj/data/engines/tcm/q{quality}/engines/{dataset}/lrp_transforms_4/fp8.engine",
        },
)

engine = build_runtime(net, codec, cfg)

xhat_blocks = torch.empty_like(blocks)
bit_len = 0
strings_bytes_list = []
state_bytes_list = []

# warmup

with torch.no_grad():
    for start in range(0, bn, batchsize):
        end = min(start + batchsize, bn)
        
        print(f"Processing blocks {start} to {end} / {bn}")
        input = blocks[start:end]   
        cur_bs = input.shape[0]
        print("input shape:", input.shape)

        if cur_bs < batchsize:
            pad_bs = batchsize - cur_bs
            pad_shape = (pad_bs,) + input.shape[1:]
            pad = torch.full(
                pad_shape,
                0.5,
                dtype=input.dtype,
                device=input.device,
            )
            input = torch.cat([input, pad], dim=0)
            print("input pad to:", input.shape)   
        
        pack,_,_ = engine.compress_time(input)
        x_hat,_,_ = engine.decompress_time(pack)
        if hasattr(codec, "pack_bytes"):
            b = codec.pack_bytes(pack)
            strings_bytes_list.append(float(b.get("strings_bytes", 0.0)))
            state_bytes_list.append(float(b.get("state_bytes", 0.0)))
        
        xhat_blocks[start:end] = x_hat[:cur_bs]
        del input, pack, x_hat
        torch.cuda.empty_cache()
        
assert xhat_blocks.shape[0] == bn
strings_bytes = sum(strings_bytes_list)
state_bytes = sum(state_bytes_list)
bit_len = strings_bytes + state_bytes
x_hat_nhw = dataloader_science.blocks_to_nhw_overlap_weighted(
    xhat_blocks,
    input_shape,
    block_size,
    stride_hw=stride,
    meta_hw=meta,
    pad_value=pad_value,
    norm_type="minmax",
    vmin=status["vmin"],
    vmax=status["vmax"],
    vmean=status["vmean"],
    vstd=status["vstd"],
)

# compute metrics
metr = metrics.basic_metrics(x_hat_nhw, x_nhw_ori)
num_pixels = x_nhw_ori.size(0) * x_nhw_ori.size(1) * x_nhw_ori.size(2)
bit_rate = bit_len * 8.0 / num_pixels

print("bpp:", bit_rate)
print("rmse:", metr["rmse"])
print("nrmse:", metr["nrmse"])
print("maxe:", metr["maxe"])
print("psnr:", metr["psnr"])

Processing blocks 0 to 34 / 34
input shape: torch.Size([34, 3, 512, 512])
[01/27/2026-13:01:06] [TRT] [W] Using default stream in enqueueV3() may lead to performance issues due to additional calls to cudaStreamSynchronize() by TensorRT to ensure correct synchronization. Please use non-default stream instead.
[01/27/2026-13:01:06] [TRT] [W] Using default stream in enqueueV3() may lead to performance issues due to additional calls to cudaStreamSynchronize() by TensorRT to ensure correct synchronization. Please use non-default stream instead.
[01/27/2026-13:01:06] [TRT] [W] Using default stream in enqueueV3() may lead to performance issues due to additional calls to cudaStreamSynchronize() by TensorRT to ensure correct synchronization. Please use non-default stream instead.
[01/27/2026-13:01:06] [TRT] [W] Using default stream in enqueueV3() may lead to performance issues due to additional calls to cudaStreamSynchronize() by TensorRT to ensure correct synchronization. Please use non-defaul

## DCAE

In [5]:
import torch
import compressai.runtime.utils.metrics as metrics
import numpy as np
import torch.nn.functional as F
from compressai.utils import dataloader_science
from compressai.models import DCAE

from compressai.runtime import build_runtime
from compressai.runtime.config import RuntimeConfig
from compressai.runtime.codecs import GpuPackedEntropyCodec
from compressai.runtime.utils.benchmark import run_e2e

# config
quality = 1
dataset = "hurricane"
bin_path = "/hwj/data/caiec_test_data/hurricane_100x1x500x500.f32"
input_shape = (100, 500, 500)            # (N,H,W)
block_size = (3, 512, 512)            # (bn,3,bh,bw)
p=0.95
stride = (int(block_size[1] * p), int(block_size[2] * p))
device = "cuda:0"
batchsize = 34
norm_type = "minmax"
pad_value = 0.5

blocks, x_nhw_ori, status, meta = dataloader_science.blockify_bin_overlap(
    bin_path, input_shape, block_size, p, stride_hw=stride,
    dtype=np.float32, pad_value=pad_value, norm_type=norm_type, device=device
)
bn = blocks.shape[0]

net = DCAE()
net = net.to(device)
net.eval()
dictory = {}

checkpoint = torch.load(f"/hwj/data/dcae/dcae-{quality}.pth", map_location=device)
for k, v in checkpoint["state_dict"].items():
    dictory[k.replace("module.", "")] = v
net.load_state_dict(dictory)
net.update()

codec = GpuPackedEntropyCodec(
    net.entropy_bottleneck,
    net.gaussian_conditional,
    P=16
)

cfg = RuntimeConfig(
    model_name="dcae",
    ga_input_dtype=torch.float16,
    gs_input_dtype=torch.float16,
    ha_input_dtype=torch.float32,
    h_z_s1_input_dtype=torch.float32,
    h_z_s2_input_dtype=torch.float16,

    dt_ca_input_dtypes = [torch.float16, torch.float16, torch.float16, torch.float16, torch.float16] ,
    cc_mean_input_dtypes = [torch.float32, torch.float32, torch.float32, torch.float32, torch.float32],
    cc_scale_input_dtypes = [torch.float32, torch.float32, torch.float32, torch.float32, torch.float32],
    lrp_input_dtypes = [torch.float32, torch.float32, torch.float32, torch.float32, torch.float32],
    
    codec_input_dtype=torch.float32,
    trt_engines={
        "ga": f"/hwj/data/engines/dcae/q{quality}/engines/{dataset}/ga/fp16.engine",
        "gs": f"/hwj/data/engines/dcae/q{quality}/engines/{dataset}/gs/fp16.engine",
        "ha": f"/hwj/data/engines/dcae/q{quality}/engines/{dataset}/ha/fp8.engine",
        "h_z_s1": f"/hwj/data/engines/dcae/q{quality}/engines/{dataset}/h_z_s1/fp8.engine",
        "h_z_s2": f"/hwj/data/engines/dcae/q{quality}/engines/{dataset}/h_z_s2/fp16.engine",

        "dt_cross_attention_0": f"/hwj/data/engines/dcae/q{quality}/engines/{dataset}/dt_cross_attention_0/fp16.engine",
        "cc_mean_0": f"/hwj/data/engines/dcae/q{quality}/engines/{dataset}/cc_mean_0/fp8.engine",
        "cc_scale_0": f"/hwj/data/engines/dcae/q{quality}/engines/{dataset}/cc_scale_0/fp8.engine",
        "lrp_transforms_0": f"/hwj/data/engines/dcae/q{quality}/engines/{dataset}/lrp_transforms_0/fp8.engine",

        "dt_cross_attention_1": f"/hwj/data/engines/dcae/q{quality}/engines/{dataset}/dt_cross_attention_1/fp16.engine",
        "cc_mean_1": f"/hwj/data/engines/dcae/q{quality}/engines/{dataset}/cc_mean_1/fp8.engine",
        "cc_scale_1": f"/hwj/data/engines/dcae/q{quality}/engines/{dataset}/cc_scale_1/fp8.engine",
        "lrp_transforms_1": f"/hwj/data/engines/dcae/q{quality}/engines/{dataset}/lrp_transforms_1/fp8.engine",

        "dt_cross_attention_2": f"/hwj/data/engines/dcae/q{quality}/engines/{dataset}/dt_cross_attention_2/fp16.engine",
        "cc_mean_2": f"/hwj/data/engines/dcae/q{quality}/engines/{dataset}/cc_mean_2/fp8.engine",
        "cc_scale_2": f"/hwj/data/engines/dcae/q{quality}/engines/{dataset}/cc_scale_2/fp8.engine",
        "lrp_transforms_2": f"/hwj/data/engines/dcae/q{quality}/engines/{dataset}/lrp_transforms_2/fp8.engine",

        "dt_cross_attention_3": f"/hwj/data/engines/dcae/q{quality}/engines/{dataset}/dt_cross_attention_3/fp16.engine",
        "cc_mean_3": f"/hwj/data/engines/dcae/q{quality}/engines/{dataset}/cc_mean_3/fp8.engine",
        "cc_scale_3": f"/hwj/data/engines/dcae/q{quality}/engines/{dataset}/cc_scale_3/fp8.engine",
        "lrp_transforms_3": f"/hwj/data/engines/dcae/q{quality}/engines/{dataset}/lrp_transforms_3/fp8.engine",

        "dt_cross_attention_4": f"/hwj/data/engines/dcae/q{quality}/engines/{dataset}/dt_cross_attention_4/fp16.engine",
        "cc_mean_4": f"/hwj/data/engines/dcae/q{quality}/engines/{dataset}/cc_mean_4/fp8.engine",
        "cc_scale_4": f"/hwj/data/engines/dcae/q{quality}/engines/{dataset}/cc_scale_4/fp8.engine",
        "lrp_transforms_4": f"/hwj/data/engines/dcae/q{quality}/engines/{dataset}/lrp_transforms_4/fp8.engine",
        }
)

engine = build_runtime(net, codec, cfg)

xhat_blocks = torch.empty_like(blocks)
bit_len = 0
strings_bytes_list = []
state_bytes_list = []

# warmup

with torch.no_grad():
    for start in range(0, bn, batchsize):
        end = min(start + batchsize, bn)
        
        print(f"Processing blocks {start} to {end} / {bn}")
        input = blocks[start:end]   
        cur_bs = input.shape[0]

        if cur_bs < batchsize:
            pad_bs = batchsize - cur_bs
            pad_shape = (pad_bs,) + input.shape[1:]
            pad = torch.full(
                pad_shape,
                0.5,
                dtype=input.dtype,
                device=input.device,
            )
            input = torch.cat([input, pad], dim=0)
            print("input pad to:", input.shape)   
              
        # print("g_a[0, 0, 0, :10]", y[0, 0, 0, :10])
        pack,_,_ = engine.compress_time(input)
        x_hat,_,_ = engine.decompress_time(pack)
        if hasattr(codec, "pack_bytes"):
            b = codec.pack_bytes(pack)
            strings_bytes_list.append(float(b.get("strings_bytes", 0.0)))
            state_bytes_list.append(float(b.get("state_bytes", 0.0)))

        xhat_blocks[start:end] = x_hat[:cur_bs]
        del input, pack, x_hat
        torch.cuda.empty_cache()
        
assert xhat_blocks.shape[0] == bn
strings_bytes = sum(strings_bytes_list)
state_bytes = sum(state_bytes_list)
bit_len = strings_bytes + state_bytes
x_hat_nhw = dataloader_science.blocks_to_nhw_overlap_weighted(
    xhat_blocks,
    input_shape,
    block_size,
    stride_hw=stride,
    meta_hw=meta,
    pad_value=pad_value,
    norm_type="minmax",
    vmin=status["vmin"],
    vmax=status["vmax"],
    vmean=status["vmean"],
    vstd=status["vstd"],
)


# compute metrics
metr = metrics.basic_metrics(x_hat_nhw, x_nhw_ori)
num_pixels = x_nhw_ori.size(0) * x_nhw_ori.size(1) * x_nhw_ori.size(2)
bit_rate = bit_len * 8.0 / num_pixels

print("bpp:", bit_rate)
print("rmse:", metr["rmse"])
print("nrmse:", metr["nrmse"])
print("maxe:", metr["maxe"])
print("psnr:", metr["psnr"])

Processing blocks 0 to 34 / 34
[01/27/2026-13:03:05] [TRT] [W] Using default stream in enqueueV3() may lead to performance issues due to additional calls to cudaStreamSynchronize() by TensorRT to ensure correct synchronization. Please use non-default stream instead.
[01/27/2026-13:03:05] [TRT] [W] Using default stream in enqueueV3() may lead to performance issues due to additional calls to cudaStreamSynchronize() by TensorRT to ensure correct synchronization. Please use non-default stream instead.
[01/27/2026-13:03:05] [TRT] [W] Using default stream in enqueueV3() may lead to performance issues due to additional calls to cudaStreamSynchronize() by TensorRT to ensure correct synchronization. Please use non-default stream instead.
[01/27/2026-13:03:05] [TRT] [W] Using default stream in enqueueV3() may lead to performance issues due to additional calls to cudaStreamSynchronize() by TensorRT to ensure correct synchronization. Please use non-default stream instead.
[01/27/2026-13:03:05] [TR